# Cointegration Strategy Testing Framework

This notebook tests baskets for cointegration sustainability and mean reversion speed.

## Testing Pipeline:
1. Load price data from parquet files
2. Generate candidate baskets (or load existing)
3. Test initial cointegration
4. Filter by sustainability across time periods
5. Filter by mean reversion speed
6. Output validated baskets for strategy deployment



In [ ]:
import sys
from pathlib import Path
import logging
from datetime import datetime, timedelta, timezone

# Find workspace root
current = Path().resolve()
while current.name != 'hku-datawork' and current.parent != current:
    current = current.parent
workspace_root = current if current.name == 'hku-datawork' else Path('/home/leo/GitHub_Clones/hku_stuff/hku-datawork')
sys.path.insert(0, str(workspace_root))

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import modules
from hypothesis_testing.cointegration.data_loader import load_price_data
from hypothesis_testing.cointegration.basket_generator import generate_baskets_clustering
from hypothesis_testing.cointegration.utils_parallel import test_baskets_cointegration_parallel
from hypothesis_testing.cointegration.filter_sustainability import filter_baskets_sustainability
from hypothesis_testing.cointegration.filter_mean_reversion import filter_baskets_mean_reversion

logger.info("Imports completed")

2025-11-05 18:55:59,021 - __main__ - INFO - Imports completed


## Configuration

Set data paths, timeframes, and filter thresholds.

In [ ]:
# Configuration
DATA_DIR = (workspace_root / "../../hku-data/test_data").resolve()
TIMEFRAME = '15m'
PRICE_TYPE = 'mark'  # 15m only available as 'mark', 1h only as 'index'
SYMBOLS = None  # None = auto-discover all symbols

# Load symbol list
sys.path.insert(0, str(workspace_root / "backtester/backtest"))
from symbol_list import symbols as symbol_list
ALL_SYMBOLS = [s.replace('-PERP', '') for s in symbol_list]
print(f"Found {len(ALL_SYMBOLS)} symbols")

# Basket generation
MIN_BASKET_SIZE = 2
MAX_BASKET_SIZE = 4
N_CLUSTERS = 5

# Sustainability filter thresholds
PERSISTENCE_THRESHOLD = 0.7  # 70% of windows/periods must pass
WINDOW_DAYS = 90
STEP_DAYS = 30
PERIOD_DAYS = 90

# Mean reversion filter thresholds
HALF_LIFE_THRESHOLD_DAYS = 30.0  # Maximum half-life in days

# Parallel processing
MAX_WORKERS = None  # None = use CPU count

# Bars per day (depends on timeframe)
BARS_PER_DAY = 96

logger.info(f"Configuration: timeframe={TIMEFRAME}, price_type={PRICE_TYPE}, "
           f"persistence_threshold={PERSISTENCE_THRESHOLD:.0%}, "
           f"half_life_threshold={HALF_LIFE_THRESHOLD_DAYS} days")

2025-11-05 18:57:09,650 - __main__ - INFO - Configuration: timeframe=15m, price_type=index, persistence_threshold=70%, half_life_threshold=30.0 days


In [ ]:
# Download 15-minute mark price data for all symbols (past 2 years)
sys.path.insert(0, str(workspace_root / "backtester/src"))
from hist_data import HistoricalDataCollector

collector = HistoricalDataCollector(data_dir=str(DATA_DIR))
end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=730)

print(f"Downloading 15m mark data: {start_date.date()} to {end_date.date()}")
print(f"Symbols: {len(ALL_SYMBOLS)}, Directory: {DATA_DIR}\n")

successful, failed = [], []
for i, symbol in enumerate(ALL_SYMBOLS, 1):
    try:
        print(f"[{i}/{len(ALL_SYMBOLS)}] {symbol}...", end=" ")
        data = collector.load_data_period(
            symbol=symbol, timeframe='15m', data_type='mark_ohlcv_futures',
            start_date=start_date, end_date=end_date, export=True
        )
        if data is not None and not data.empty:
            successful.append(symbol)
            print(f"✓ ({len(data):,} records)")
        else:
            failed.append(symbol)
            print("✗")
    except Exception as e:
        failed.append(symbol)
        print(f"✗ ({str(e)[:40]})")

print(f"\nComplete: {len(successful)} successful, {len(failed)} failed")


## Step 1: Load Price Data

Load historical price data from parquet files.

In [8]:
price_data = load_price_data(
    data_dir=DATA_DIR,
    symbols=SYMBOLS,
    timeframe=TIMEFRAME,
    price_type=PRICE_TYPE,
    max_workers=MAX_WORKERS
)

logger.info(f"Loaded price data: {len(price_data)} timestamps, {len(price_data.columns)} symbols")
logger.info(f"Date range: {price_data.index.min()} to {price_data.index.max()}")
price_data.head()

2025-11-05 18:57:10,951 - hypothesis_testing.cointegration.data_loader - INFO - Loading data from 50 parquet files for 10 symbols...
2025-11-05 18:57:11,264 - hypothesis_testing.cointegration.data_loader - INFO - Aligning timestamps across all symbols...
2025-11-05 18:57:11,287 - hypothesis_testing.cointegration.data_loader - INFO - Loaded 20092 timestamps for 2 symbols
2025-11-05 18:57:11,288 - hypothesis_testing.cointegration.data_loader - INFO - Date range: 2025-03-16 07:15:00+00:00 to 2025-10-11 14:00:00+00:00
2025-11-05 18:57:11,289 - __main__ - INFO - Loaded price data: 20092 timestamps, 2 symbols
2025-11-05 18:57:11,289 - __main__ - INFO - Date range: 2025-03-16 07:15:00+00:00 to 2025-10-11 14:00:00+00:00


,NEAR_close,OXT_close
timestamp,,
2025-03-16 07:15:00+00:00,2.646054,0.074771
2025-03-16 07:30:00+00:00,2.659189,0.074796
2025-03-16 07:45:00+00:00,2.656107,0.074716
2025-03-16 08:00:00+00:00,2.650043,0.074677
2025-03-16 08:15:00+00:00,2.644343,0.074567


## Step 2: Generate Candidate Baskets

Generate candidate baskets using clustering (or load existing baskets).

In [ ]:
# Extract base symbols (remove _close suffix)
base_symbols = [col.replace('_close', '') for col in price_data.columns]

candidate_baskets = generate_baskets_clustering(
    price_data=price_data,
    n_clusters=N_CLUSTERS,
    min_basket_size=MIN_BASKET_SIZE,
    max_basket_size=MAX_BASKET_SIZE,
    parallel=True,
    max_workers=MAX_WORKERS
)

logger.info(f"Generated {len(candidate_baskets)} candidate baskets")
print(f"Sample baskets: {candidate_baskets[:5]}")

## Step 3: Test Initial Cointegration

Test all candidate baskets for cointegration using Johansen test (multiprocessed).

In [ ]:
cointegrated_baskets = test_baskets_cointegration_parallel(
    price_data=price_data,
    candidate_baskets=candidate_baskets,
    max_workers=MAX_WORKERS,
    batch_size=100
)

logger.info(f"Found {len(cointegrated_baskets)} cointegrated baskets out of {len(candidate_baskets)} candidates")
print(f"Cointegration rate: {len(cointegrated_baskets)/len(candidate_baskets):.1%}")

## Step 4: Filter by Sustainability

Test cointegration persistence across rolling windows and discrete periods. 
This ensures the cointegration relationship is stable over time (not just in-sample).

In [ ]:
sustainable_baskets = filter_baskets_sustainability(
    baskets=cointegrated_baskets,
    price_data=price_data,
    persistence_threshold=PERSISTENCE_THRESHOLD,
    window_days=WINDOW_DAYS,
    step_days=STEP_DAYS,
    period_days=PERIOD_DAYS,
    bars_per_day=BARS_PER_DAY,
    use_rolling=True,
    use_discrete=True,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(sustainable_baskets)} sustainable baskets "
           f"(from {len(cointegrated_baskets)} cointegrated)")

# Display sample results
if sustainable_baskets:
    sample = sustainable_baskets[0]
    print(f"\nSample sustainable basket: {sample['basket']}")
    print(f"Rolling windows persistence: {sample['sustainability']['rolling_windows']['persistence_ratio']:.1%}")
    print(f"Discrete periods persistence: {sample['sustainability']['discrete_periods']['persistence_ratio']:.1%}")

## Step 5: Filter by Mean Reversion Speed

Filter for baskets that mean revert quickly (important for trading performance).
This tests whether the spread reverts fast enough to be profitable.

In [ ]:
fast_mean_reverting_baskets = filter_baskets_mean_reversion(
    baskets=sustainable_baskets,
    half_life_threshold_days=HALF_LIFE_THRESHOLD_DAYS,
    bars_per_day=BARS_PER_DAY,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(fast_mean_reverting_baskets)} fast mean-reverting baskets "
           f"(from {len(sustainable_baskets)} sustainable)")

# Display sample results
if fast_mean_reverting_baskets:
    sample = fast_mean_reverting_baskets[0]
    print(f"\nSample fast mean-reverting basket: {sample['basket']}")
    print(f"Half-life: {sample['mean_reversion']['half_life_days']:.1f} days")
    print(f"ADF p-value: {sample['mean_reversion']['adf_p_value']:.4f}")
    print(f"Is stationary: {sample['mean_reversion']['is_stationary']}")

## Step 6: Save Validated Baskets

Save the proven baskets for strategy deployment.

In [ ]:
import json

# Prepare baskets for saving (convert numpy arrays to lists)
validated_baskets_output = []
for basket_result in fast_mean_reverting_baskets:
    output = {
        'basket': basket_result['basket'],
        'eigenvector': basket_result['eigenvector'].tolist(),
        'johansen_p_value': basket_result['johansen_result']['p_value'],
        'johansen_trace_stat': float(basket_result['johansen_result']['trace_stat']),
        'sustainability_rolling': basket_result['sustainability']['rolling_windows']['persistence_ratio'],
        'sustainability_discrete': basket_result['sustainability']['discrete_periods']['persistence_ratio'],
        'half_life_days': float(basket_result['mean_reversion']['half_life_days']),
        'adf_p_value': float(basket_result['mean_reversion']['adf_p_value']),
        'is_stationary': basket_result['mean_reversion']['is_stationary']
    }
    validated_baskets_output.append(output)

# Save to JSON
output_file = Path("validated_baskets.json")
with open(output_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'config': {
            'timeframe': TIMEFRAME,
            'price_type': PRICE_TYPE,
            'persistence_threshold': PERSISTENCE_THRESHOLD,
            'half_life_threshold_days': HALF_LIFE_THRESHOLD_DAYS
        },
        'baskets': validated_baskets_output
    }, f, indent=2)

logger.info(f"Saved {len(validated_baskets_output)} validated baskets to {output_file}")

# Display summary
print(f"\n{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total candidate baskets: {len(candidate_baskets)}")
print(f"Cointegrated baskets: {len(cointegrated_baskets)} ({len(cointegrated_baskets)/len(candidate_baskets):.1%})")
print(f"Sustainable baskets: {len(sustainable_baskets)} ({len(sustainable_baskets)/len(cointegrated_baskets):.1%} of cointegrated)")
print(f"Fast mean-reverting baskets: {len(fast_mean_reverting_baskets)} ({len(fast_mean_reverting_baskets)/len(sustainable_baskets):.1%} of sustainable)")
print(f"{'='*60}")
print(f"\nValidated baskets ready for strategy deployment!")
print(f"Output file: {output_file}")